In [1]:
import pandas as pd
import numpy as np


In [2]:
from google.colab import files
uploaded = files.upload()

Saving bank_dataset_1.csv to bank_dataset_1.csv


In [3]:
df = pd.read_csv("bank_dataset_1.csv")
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

Dataset shape: (10000, 200)

First 5 rows:
   customer_id   age  gender marital_status education_level  annual_income  \
0    427741615  30.0  Female        Widowed          Master       14073.58   
1    228500249  35.0    Male         Single        Bachelor            NaN   
2    396233462  59.0    Male        Married          Master       16764.51   
3    909211459  26.0    Male        Married          Master       53460.55   
4    982980455  59.0    Male         Single        Bachelor       66520.40   

   loan_amount  credit_score  employment_years employment_type  ...  \
0    147079.61         803.0              29.5   Self-employed  ...   
1     14771.04         577.0              33.6       Part-time  ...   
2    194804.70         790.0              40.0       Full-time  ...   
3    138636.74         684.0              27.7       Part-time  ...   
4     79776.32         662.0              37.7       Full-time  ...   

  flag_feature_173 category_feature_174  numeric_feature_175 

In [4]:
print("\n--- Data types ---")
print(df.dtypes.value_counts())

print("\n--- Column list with dtypes ---")
print(df.dtypes)


--- Data types ---
float64    149
object      50
int64        1
Name: count, dtype: int64

--- Column list with dtypes ---
customer_id               int64
age                     float64
gender                   object
marital_status           object
education_level          object
                         ...   
flag_feature_178        float64
flag_feature_179        float64
category_feature_180     object
flag_feature_181        float64
category_feature_182     object
Length: 200, dtype: object


In [5]:
# Missing value profiling
missing_count = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct,
    'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

print("\n--- Missing value report (top 30) ---")
print(missing_report.head(30))


--- Missing value report (top 30) ---
                      missing_count  missing_pct    dtype
ratio_feature_104               806         8.06  float64
numeric_feature_76              784         7.84  float64
numeric_feature_175             762         7.62  float64
numeric_feature_86              751         7.51  float64
flag_feature_117                749         7.49  float64
numeric_feature_56              748         7.48  float64
category_feature_172            748         7.48   object
numeric_feature_25              746         7.46  float64
ratio_feature_17                745         7.45  float64
flag_feature_167                744         7.44  float64
category_feature_49             744         7.44   object
numeric_feature_37              743         7.43  float64
flag_feature_81                 742         7.42  float64
numeric_feature_125             742         7.42  float64
numeric_feature_134             741         7.41  float64
flag_feature_79                 7

In [6]:
no_missing_cols = df.columns[df.isnull().sum() == 0]

print("Columns with no missing values:")
for col in no_missing_cols:
    print(col)

Columns with no missing values:
customer_id


In [7]:
missing_report.to_csv('missing_value_report.csv')
print("\nFull missing value report saved to missing_value_report.csv")



Full missing value report saved to missing_value_report.csv


In [8]:
missing_pct = (df.isnull().mean() * 100).round(2)

above_5 = (missing_pct > 5).sum()
below_or_equal_5 = (missing_pct <= 5).sum()

print(f"Columns with missing % > 5%: {above_5}")
print(f"Columns with missing % <= 5%: {below_or_equal_5}")

Columns with missing % > 5%: 199
Columns with missing % <= 5%: 1


In [9]:
drop_cols = missing_report[missing_report['missing_pct'] > 55].index.tolist()
investigate_cols = missing_report[
    (missing_report['missing_pct'] > 5) & (missing_report['missing_pct'] <= 55)
].index.tolist()
keep_cols = missing_report[missing_report['missing_pct'] <= 5].index.tolist()

print(f"\nColumns to consider DROPPING (>55% missing): {len(drop_cols)}")
print(drop_cols)

print(f"\nColumns to INVESTIGATE/IMPUTE (5-55% missing): {len(investigate_cols)}")
print(investigate_cols)

print(f"\nColumns to KEEP as-is (<5% missing): {len(keep_cols)}")


Columns to consider DROPPING (>55% missing): 0
[]

Columns to INVESTIGATE/IMPUTE (5-55% missing): 199
['ratio_feature_104', 'numeric_feature_76', 'numeric_feature_175', 'numeric_feature_86', 'flag_feature_117', 'numeric_feature_56', 'category_feature_172', 'numeric_feature_25', 'ratio_feature_17', 'flag_feature_167', 'category_feature_49', 'numeric_feature_37', 'flag_feature_81', 'numeric_feature_125', 'numeric_feature_134', 'flag_feature_79', 'flag_feature_45', 'numeric_feature_33', 'flag_feature_77', 'numeric_feature_143', 'category_feature_91', 'application_date', 'numeric_feature_36', 'category_feature_92', 'category_feature_61', 'numeric_feature_50', 'interest_rate', 'ratio_feature_13', 'category_feature_39', 'category_feature_107', 'debt_to_income_ratio', 'flag_feature_122', 'numeric_feature_57', 'existing_loans', 'category_feature_47', 'annual_income', 'flag_feature_66', 'ratio_feature_99', 'credit_score', 'ratio_feature_3', 'flag_feature_155', 'ratio_feature_126', 'employment_

In [10]:
# Duplicate check
print("\n--- Duplicate rows ---")
print("Full row duplicates:", df.duplicated().sum())


--- Duplicate rows ---
Full row duplicates: 0


In [11]:
#find columns whose names contain "id".
id_col_candidates = [c for c in df.columns if 'id' in c.lower()]
print("Possible ID columns:", id_col_candidates)

Possible ID columns: ['customer_id']


In [12]:
cat_cols = df.select_dtypes(include=['object']).columns

print("\n--- Unique value counts for categorical columns (first 20) ---")
for col in cat_cols[:20]:
    n_unique = df[col].nunique()
    print(f"\n{col}: {n_unique} unique values")
    if n_unique <= 15:
        print(df[col].value_counts(dropna=False))



--- Unique value counts for categorical columns (first 20) ---

gender: 3 unique values
gender
Female    4621
Male      4507
NaN        700
Other      172
Name: count, dtype: int64

marital_status: 4 unique values
marital_status
Single      4208
Married     3694
Divorced     941
NaN          721
Widowed      436
Name: count, dtype: int64

education_level: 5 unique values
education_level
Bachelor       3546
High School    2973
Master         1673
NaN             693
Diploma         678
PhD             437
Name: count, dtype: int64

employment_type: 4 unique values
employment_type
Full-time        5054
Part-time        1907
Self-employed    1389
Unemployed        952
NaN               698
Name: count, dtype: int64

loan_purpose: 6 unique values
loan_purpose
Home         2790
Personal     1881
Car          1851
Business     1336
Education     951
NaN           701
Medical       490
Name: count, dtype: int64

home_ownership: 4 unique values
home_ownership
Own         3643
Mortgage    3246

In [13]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("\n--- Numeric summary stats ---")
print(df[num_cols].describe().T)



--- Numeric summary stats ---
                       count          mean           std           min  \
customer_id          10000.0  5.460793e+08  2.594427e+08  1.000280e+08   
age                   9290.0  4.602110e+01  1.646836e+01  1.800000e+01   
annual_income         9275.0  3.957036e+04  2.627941e+04  3.240190e+03   
loan_amount           9316.0  2.498059e+05  1.444095e+05  1.066510e+03   
credit_score          9277.0  6.780572e+02  8.357993e+01  3.710000e+02   
...                      ...           ...           ...           ...   
flag_feature_176      9340.0  3.036403e-01  4.598538e-01  0.000000e+00   
numeric_feature_177   9297.0  9.976051e+02  3.027127e+02 -1.691600e+02   
flag_feature_178      9283.0  2.994722e-01  4.580515e-01  0.000000e+00   
flag_feature_179      9300.0  3.004301e-01  4.584697e-01  0.000000e+00   
flag_feature_181      9282.0  2.985348e-01  4.576400e-01  0.000000e+00   

                              25%           50%           75%           max  
cu

In [14]:
print("\n--- Checking for numeric-looking text columns ---")
for col in cat_cols:
    sample_vals = df[col].dropna().astype(str).head(50)
    numeric_like = sample_vals.str.replace('.', '', regex=False).str.replace('-', '', regex=False).str.isnumeric().mean()
    if numeric_like > 0.7:
        print(f"{col}: looks numeric but stored as text ({numeric_like*100:.0f}% of sample)")


--- Checking for numeric-looking text columns ---
application_date: looks numeric but stored as text (100% of sample)


In [15]:
print(df.columns.tolist())

['customer_id', 'age', 'gender', 'marital_status', 'education_level', 'annual_income', 'loan_amount', 'credit_score', 'employment_years', 'employment_type', 'loan_purpose', 'home_ownership', 'debt_to_income_ratio', 'existing_loans', 'loan_term_months', 'interest_rate', 'loan_status', 'application_date', 'flag_feature_1', 'category_feature_2', 'ratio_feature_3', 'numeric_feature_4', 'numeric_feature_5', 'numeric_feature_6', 'flag_feature_7', 'numeric_feature_8', 'category_feature_9', 'numeric_feature_10', 'numeric_feature_11', 'ratio_feature_12', 'ratio_feature_13', 'numeric_feature_14', 'category_feature_15', 'numeric_feature_16', 'ratio_feature_17', 'numeric_feature_18', 'numeric_feature_19', 'category_feature_20', 'numeric_feature_21', 'ratio_feature_22', 'numeric_feature_23', 'category_feature_24', 'numeric_feature_25', 'category_feature_26', 'flag_feature_27', 'ratio_feature_28', 'category_feature_29', 'numeric_feature_30', 'flag_feature_31', 'category_feature_32', 'numeric_feature

In [16]:
business_relevant_cols = [
    'customer_id',
    'age',
    'gender',
    'marital_status',
    'education_level',
    'annual_income',
    'loan_amount',
    'credit_score',
    'employment_years',
    'employment_type',
    'loan_purpose',
    'home_ownership',
    'debt_to_income_ratio',
    'existing_loans',
    'loan_term_months',
    'interest_rate',
    'loan_status',       #target column
    'application_date'
]

In [17]:
df['loan_status'].unique()
df['loan_status'].value_counts()

,count
loan_status,
Approved,6743
Rejected,2581


In [18]:
df['loan_status'].isnull().sum()
df['loan_status'].value_counts(dropna=False)

,count
loan_status,
Approved,6743
Rejected,2581
NaN,676


In [19]:
business_relevant_cols = ['customer_id', 'age', 'gender', 'marital_status', 'education_level',
                           'annual_income', 'loan_amount', 'credit_score', 'employment_years',
                           'employment_type', 'loan_purpose', 'home_ownership',
                           'debt_to_income_ratio', 'existing_loans', 'loan_term_months',
                           'interest_rate', 'loan_status', 'application_date']

if business_relevant_cols:
    df_focus = df[business_relevant_cols]
    df_focus.to_csv('focused_dataset.csv', index=False)
    print("\nFocused dataset exported to focused_dataset.csv")


Focused dataset exported to focused_dataset.csv


In [20]:
relevant_cols = ['customer_id', 'age', 'gender', 'marital_status', 'education_level',
                  'annual_income', 'loan_amount', 'credit_score', 'employment_years',
                  'employment_type', 'loan_purpose', 'home_ownership',
                  'debt_to_income_ratio', 'existing_loans', 'loan_term_months',
                  'interest_rate', 'loan_status', 'application_date']

# Missing values in just these columns
print(df[relevant_cols].isnull().sum())

# Data types
print(df[relevant_cols].dtypes)

# Quick look at the categorical ones
for col in ['gender', 'marital_status', 'education_level', 'employment_type',
            'loan_purpose', 'home_ownership']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

customer_id               0
age                     710
gender                  700
marital_status          721
education_level         693
annual_income           725
loan_amount             684
credit_score            723
employment_years        722
employment_type         698
loan_purpose            701
home_ownership          718
debt_to_income_ratio    726
existing_loans          725
loan_term_months        702
interest_rate           730
loan_status             676
application_date        735
dtype: int64
customer_id               int64
age                     float64
gender                   object
marital_status           object
education_level          object
annual_income           float64
loan_amount             float64
credit_score            float64
employment_years        float64
employment_type          object
loan_purpose             object
home_ownership           object
debt_to_income_ratio    float64
existing_loans          float64
loan_term_months        float64
int

In [21]:
print(df[['age', 'annual_income', 'loan_amount', 'credit_score',
          'employment_years', 'debt_to_income_ratio', 'existing_loans',
          'loan_term_months', 'interest_rate']].describe())

               age  annual_income    loan_amount  credit_score  \
count  9290.000000    9275.000000    9316.000000   9277.000000   
mean     46.021098   39570.355780  249805.891756    678.057238   
std      16.468364   26279.408111  144409.498380     83.579935   
min      18.000000    3240.190000    1066.510000    371.000000   
25%      32.000000   22038.360000  123805.420000    621.000000   
50%      46.000000   33169.010000  247723.575000    679.000000   
75%      60.000000   49248.985000  377296.127500    735.000000   
max      74.000000  329105.270000  499964.710000    850.000000   

       employment_years  debt_to_income_ratio  existing_loans  \
count       9278.000000           9274.000000     9275.000000   
mean          19.888640              0.434236        1.402049   
std           11.583725              0.240208        1.180212   
min            0.000000              0.010000        0.000000   
25%            9.700000              0.228000        1.000000   
50%           1

In [22]:
relevant_cols = ['customer_id', 'age', 'gender', 'marital_status', 'education_level',
                  'annual_income', 'loan_amount', 'credit_score', 'employment_years',
                  'employment_type', 'loan_purpose', 'home_ownership',
                  'debt_to_income_ratio', 'existing_loans', 'loan_term_months',
                  'interest_rate', 'loan_status', 'application_date']

df_focus = df[relevant_cols].copy()

print("Starting shape:", df_focus.shape)

Starting shape: (10000, 18)


In [23]:
df_focus = df_focus.dropna(subset=['loan_status'])
print("After dropping missing loan_status:", df_focus.shape)


After dropping missing loan_status: (9324, 18)


In [24]:
# Handle missing values - NUMERIC columns (impute with median)

numeric_cols = ['age', 'annual_income', 'loan_amount', 'credit_score',
                 'employment_years', 'debt_to_income_ratio', 'existing_loans',
                 'loan_term_months', 'interest_rate']

for col in numeric_cols:
    median_val = df_focus[col].median()
    df_focus[col] = df_focus[col].fillna(median_val)
    print(f"{col}: filled missing with median = {median_val:.2f}")


age: filled missing with median = 46.00
annual_income: filled missing with median = 33257.31
loan_amount: filled missing with median = 247530.11
credit_score: filled missing with median = 679.00
employment_years: filled missing with median = 19.70
debt_to_income_ratio: filled missing with median = 0.43
existing_loans: filled missing with median = 1.00
loan_term_months: filled missing with median = 60.00
interest_rate: filled missing with median = 13.88


In [25]:
#4 Handle missing values - CATEGORICAL columns
categorical_cols = ['gender', 'marital_status', 'education_level',
                     'employment_type', 'loan_purpose', 'home_ownership']

for col in categorical_cols:
    df_focus[col] = df_focus[col].fillna('Unknown')
    print(f"{col}: filled missing with 'Unknown'")


gender: filled missing with 'Unknown'
marital_status: filled missing with 'Unknown'
education_level: filled missing with 'Unknown'
employment_type: filled missing with 'Unknown'
loan_purpose: filled missing with 'Unknown'
home_ownership: filled missing with 'Unknown'


In [26]:
#Handle application_date
df_focus['application_date'] = pd.to_datetime(df_focus['application_date'], errors='coerce')
missing_dates = df_focus['application_date'].isnull().sum()
print(f"\napplication_date: {missing_dates} rows have missing/unparseable dates "
      f"(kept in dataset, will be excluded only from date-based charts)")

# Extract useful date parts for later trend analysis
df_focus['application_year'] = df_focus['application_date'].dt.year
df_focus['application_month'] = df_focus['application_date'].dt.month


application_date: 696 rows have missing/unparseable dates (kept in dataset, will be excluded only from date-based charts)


/tmp/ipykernel_3214/2201440382.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_focus['application_date'] = pd.to_datetime(df_focus['application_date'], errors='coerce')


In [27]:
df_focus['customer_id'] = df_focus['customer_id'].astype(int)
df_focus['age'] = df_focus['age'].round(0).astype(int)
df_focus['employment_years'] = df_focus['employment_years'].round(1)
df_focus['existing_loans'] = df_focus['existing_loans'].round(0).astype(int)
df_focus['loan_term_months'] = df_focus['loan_term_months'].round(0).astype(int)


In [28]:
for col in categorical_cols:
    df_focus[col] = df_focus[col].str.strip().str.title()

df_focus['loan_status'] = df_focus['loan_status'].str.strip().str.title()



In [29]:
print("\n--- Final missing value check ---")
print(df_focus.isnull().sum())

print("\n--- Final shape ---")
print(df_focus.shape)

print("\n--- Final dtypes ---")
print(df_focus.dtypes)

print("\n--- loan_status distribution ---")
print(df_focus['loan_status'].value_counts())



--- Final missing value check ---
customer_id               0
age                       0
gender                    0
marital_status            0
education_level           0
annual_income             0
loan_amount               0
credit_score              0
employment_years          0
employment_type           0
loan_purpose              0
home_ownership            0
debt_to_income_ratio      0
existing_loans            0
loan_term_months          0
interest_rate             0
loan_status               0
application_date        696
application_year        696
application_month       696
dtype: int64

--- Final shape ---
(9324, 20)

--- Final dtypes ---
customer_id                      int64
age                              int64
gender                          object
marital_status                  object
education_level                 object
annual_income                  float64
loan_amount                    float64
credit_score                   float64
employment_years          

In [30]:
df_focus['income_bracket'] = pd.cut(
    df_focus['annual_income'],
    bins=[0, 20000, 40000, 60000, np.inf],
    labels=['Low (<20K)', 'Lower-Mid (20-40K)', 'Upper-Mid (40-60K)', 'High (60K+)']
)
df_focus['credit_score_band'] = pd.cut(
    df_focus['credit_score'],
    bins=[0, 580, 670, 740, 800, 850],
    labels=['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']
)



In [31]:
# Export cleaned dataset
df_focus.to_csv('cleaned_loan_data.csv', index=False)
print("\n=== CLEANING COMPLETE ===")
print("Cleaned dataset exported to cleaned_loan_data.csv")
print(f"Final dataset: {df_focus.shape[0]} rows x {df_focus.shape[1]} columns")
print("\nNext step: load cleaned_loan_data.csv into SQL for business-question analysis")


=== CLEANING COMPLETE ===
Cleaned dataset exported to cleaned_loan_data.csv
Final dataset: 9324 rows x 22 columns

Next step: load cleaned_loan_data.csv into SQL for business-question analysis
